# Machine Learning Cuántico: Kernels

**Objetivo:** implementar un kernel cuántico y compararlo con el kernel RBF clásico.

La idea del kernel cuántico: mapear datos al espacio de Hilbert de n qubits con un feature map cuántico φ(x), y usar el producto interno ⟨φ(x)|φ(x')⟩ como función de similaridad para SVM.

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.random.seed(42)

# Dataset XOR cuántico
n_samples = 60
X = np.random.randn(n_samples, 2)
y = (X[:,0]*X[:,1] > 0).astype(int)  # XOR lineal

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {len(X_train)} muestras, Test: {len(X_test)} muestras')

In [ ]:
# Feature Map Cuántico: ZZFeatureMap manual (2 qubits)
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def quantum_feature_map(x):
    """ZZFeatureMap: H + rotaciones Z + ZZ cruzadas"""
    n = len(x)
    qc = QuantumCircuit(n)
    # Capa 1: Hadamard + RZ
    qc.h(range(n))
    for i in range(n):
        qc.rz(2 * x[i], i)
    # Capa 2: CNOT + RZ cruzadas
    qc.cx(0, 1)
    qc.rz(2 * (np.pi - x[0]) * (np.pi - x[1]), 1)
    qc.cx(0, 1)
    return qc

# Kernel cuántico: K(x,x') = |⟨φ(x)|φ(x')⟩|²
def quantum_kernel_entry(x1, x2):
    qc1 = quantum_feature_map(x1)
    qc2 = quantum_feature_map(x2)
    sv1 = Statevector.from_instruction(qc1)
    sv2 = Statevector.from_instruction(qc2)
    return abs(sv1.data.conj() @ sv2.data)**2

# Ejemplo
x1, x2 = X_train[0], X_train[1]
print(f'K(x1,x1) = {quantum_kernel_entry(x1,x1):.4f}  (debe ser 1.0)')
print(f'K(x1,x2) = {quantum_kernel_entry(x1,x2):.4f}')

In [ ]:
# Calcular matriz de Gram para subconjunto (lento sin aceleración)
N_small = 20  # usar subconjunto para velocidad
X_tr_s = X_train[:N_small]
y_tr_s = y_train[:N_small]
X_te_s = X_test[:10]
y_te_s = y_test[:10]

print('Calculando kernel cuántico...')
K_train = np.zeros((N_small, N_small))
for i in range(N_small):
    for j in range(i, N_small):
        k = quantum_kernel_entry(X_tr_s[i], X_tr_s[j])
        K_train[i,j] = k
        K_train[j,i] = k
print('Kernel matrix shape:', K_train.shape)

In [ ]:
# SVM con kernel cuántico
svm_q = SVC(kernel='precomputed', C=1.0)
svm_q.fit(K_train, y_tr_s)

K_test = np.array([[quantum_kernel_entry(X_te_s[i], X_tr_s[j])
                     for j in range(N_small)] for i in range(10)])
acc_q = accuracy_score(y_te_s, svm_q.predict(K_test))

# Comparar con RBF clásico
svm_rbf = SVC(kernel='rbf', C=1.0)
svm_rbf.fit(X_tr_s, y_tr_s)
acc_rbf = accuracy_score(y_te_s, svm_rbf.predict(X_te_s))

print(f'Accuracy — Kernel cuántico: {acc_q:.2f},  RBF clásico: {acc_rbf:.2f}')

## Kernel Target Alignment (KTA)

KTA mide cuán bien alineado está el kernel con las etiquetas:
KTA = ⟨K, yy^T⟩_F / (||K||_F · ||yy^T||_F)

In [ ]:
def kta(K, y):
    Y = np.outer(2*y-1, 2*y-1)  # +1/-1 labels
    return np.sum(K*Y) / (np.linalg.norm(K,'fro') * np.linalg.norm(Y,'fro'))

kta_q = kta(K_train, y_tr_s)
print(f'KTA cuántico: {kta_q:.4f}')

# RBF kernel
from sklearn.metrics.pairwise import rbf_kernel
K_rbf = rbf_kernel(X_tr_s)
kta_rbf = kta(K_rbf, y_tr_s)
print(f'KTA RBF:      {kta_rbf:.4f}')

## Próximos pasos

- **Lab:** `Cuadernos/laboratorios/23_qml_kernel_alignment.ipynb`
- **QML completo:** `Cuadernos/laboratorios/40_qml_datos_reales.ipynb`
- **Siguiente guía:** `14_computacion_adiabatica.ipynb`